# 02. Exportação dos artefatos da reprodução rápida

Este notebook exporta os artefatos produzidos pela reprodução rápida em `pi-defense-rpdct`.

A reprodução rápida foi criada para baixar os artefatos do Hugging Face, executar uma amostra pequena do experimento e gerar métricas compactas. Esta etapa final organiza esses resultados em um pacote leve de exportação.

A lógica deste notebook segue a mesma decisão usada no projeto principal: **não copiar arquivos grandes para uma pasta intermediária**. Em vez disso, o notebook cria apenas metadados leves dentro de `exports/` e monta o arquivo `.zip` lendo os arquivos diretamente de seus caminhos originais.

Isso evita duplicar dados, resultados e arquivos baixados do Hugging Face, o que é importante em ambientes com pouco espaço em disco.

## 1. Estrutura esperada

Este notebook espera que a reprodução rápida esteja organizada assim:

```text
/workspace/
  pi-defense-rpdct/
    .venv/
    requirements-rpdct.txt
    notebook/
      01_reproduce_from_huggingface.ipynb
      02_export_reproduction_artifacts.ipynb
    data/
    results/
    logs/
    manifests/
    exports/
  zip_files/
```

A pasta `pi-defense-rpdct` é independente do projeto principal `pi-defense-exp`. Ela possui seu próprio ambiente virtual, seu próprio notebook de reprodução e seus próprios resultados.

Este notebook gera metadados leves em:

```text
/workspace/pi-defense-rpdct/exports/reproduction_artifacts/<run_mode>/
```

E salva o arquivo compactado final em:

```text
/workspace/zip_files/
```

Assim, o `.zip` não fica dentro da própria pasta que está sendo indexada e não causa duplicação desnecessária.

In [ ]:
from pathlib import Path

RPDCT_ROOT = Path("/workspace/pi-defense-rpdct")
NOTEBOOK_DIR = RPDCT_ROOT / "notebook"
ZIP_FILES_DIR = Path("/workspace/zip_files")

RUN_MODE = "quick"

EXPORT_ROOT = RPDCT_ROOT / "exports" / "reproduction_artifacts" / RUN_MODE
EXPORT_INDEX_DIR = EXPORT_ROOT / "index"

for directory in [RPDCT_ROOT, NOTEBOOK_DIR, ZIP_FILES_DIR, EXPORT_ROOT, EXPORT_INDEX_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("RPDCT root:", RPDCT_ROOT)
print("Notebook dir:", NOTEBOOK_DIR)
print("Export metadata root:", EXPORT_ROOT)
print("Zip files dir:", ZIP_FILES_DIR)

## 2. Política de exportação

A exportação separa três tipos de artefatos.

O primeiro grupo são os artefatos que devem entrar no pacote final, como notebooks, requirements, logs, manifestos e resultados da reprodução. Eles serão lidos diretamente dos caminhos originais e adicionados ao `.zip`.

O segundo grupo são arquivos opcionais que podem não existir. Se uma pasta opcional não existir, isso será registrado em `missing_optional_sources.json`, mas não será tratado como erro.

O terceiro grupo são arquivos ignorados por política, principalmente porque podem ser grandes ou desnecessários para auditoria rápida. Por padrão, o notebook não exporta caches do Hugging Face, ambientes virtuais ou snapshots completos baixados de modelos/datasets.

A regra principal é:

```text
Não copiar artefatos pesados para exports/.
Criar apenas metadados leves em exports/.
Gerar o zip lendo diretamente os arquivos originais.
```

In [ ]:
# Fontes leves que devem ser incluídas quando existirem.
EXPORT_REQUIREMENTS = True
EXPORT_NOTEBOOKS = True
EXPORT_RESULTS = True
EXPORT_LOGS = True
EXPORT_MANIFESTS = True
EXPORT_SAMPLE_DATA = True
EXPORT_METADATA = True

# Fontes potencialmente grandes. Ficam desativadas por padrão.
EXPORT_HF_CACHE = False
EXPORT_ADAPTER_CACHE = False
EXPORT_DOWNLOADED_HF_SNAPSHOTS = False
EXPORT_VENV = False

# Limite de segurança por arquivo. Arquivos maiores que isso serão ignorados,
# a menos que você aumente explicitamente o limite.
MAX_FILE_SIZE_MB = 1024

ARCHIVE_BASENAME = f"pi_defense_rpdct_artifacts_{RUN_MODE}"
ARCHIVE_PATH = ZIP_FILES_DIR / f"{ARCHIVE_BASENAME}.zip"
ARCHIVE_INFO_PATH = ZIP_FILES_DIR / f"{ARCHIVE_BASENAME}_zip_info.json"

print("Archive path:", ARCHIVE_PATH)
print("Max file size MB:", MAX_FILE_SIZE_MB)

## 3. Funções utilitárias

As funções abaixo padronizam escrita de JSON, cálculo de hash, listagem de arquivos e montagem do índice de exportação.

O índice é importante porque permite saber exatamente quais arquivos foram incluídos no `.zip`, de onde vieram no sistema de arquivos e qual será o caminho interno deles dentro do arquivo compactado.

In [ ]:
import hashlib
import json
import math
import zipfile
from datetime import datetime, timezone
from typing import Any

import pandas as pd


def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat()


def sanitize_json_value(value: Any) -> Any:
    if isinstance(value, dict):
        return {str(k): sanitize_json_value(v) for k, v in value.items()}
    if isinstance(value, list):
        return [sanitize_json_value(v) for v in value]
    if isinstance(value, tuple):
        return [sanitize_json_value(v) for v in value]
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, float):
        if math.isnan(value):
            return "NaN"
        if math.isinf(value):
            return "Infinity" if value > 0 else "-Infinity"
        return value
    return value


def write_json(path: Path, data: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(sanitize_json_value(data), f, indent=2, ensure_ascii=False, allow_nan=False)


def count_jsonl_lines(path: Path) -> int:
    if not path.exists():
        return 0
    with open(path, "r", encoding="utf-8") as f:
        return sum(1 for line in f if line.strip())


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def should_skip_file(path: Path) -> tuple[bool, str | None]:
    parts = set(path.parts)
    name = path.name

    if ".venv" in parts:
        return True, "virtual_environment"
    if "__pycache__" in parts:
        return True, "python_cache"
    if ".ipynb_checkpoints" in parts:
        return True, "jupyter_checkpoint"
    if name.endswith(".pyc"):
        return True, "python_bytecode"
    if name.endswith(".tmp") or name.endswith(".temp"):
        return True, "temporary_file"

    if "hf_cache" in parts and not EXPORT_HF_CACHE:
        return True, "hf_cache_disabled"
    if "adapter_cache" in parts and not EXPORT_ADAPTER_CACHE:
        return True, "adapter_cache_disabled"

    size_mb = path.stat().st_size / (1024 * 1024)
    if size_mb > MAX_FILE_SIZE_MB:
        return True, f"file_larger_than_{MAX_FILE_SIZE_MB}_mb"

    return False, None


def iter_files(source: Path):
    if source.is_file():
        yield source
    elif source.is_dir():
        for path in sorted(source.rglob("*")):
            if path.is_file():
                yield path

## 4. Fontes consideradas para exportação

Nesta etapa, o notebook define quais grupos de arquivos serão considerados.

Os grupos principais são:

```text
requirements
notebooks
sample_data
results
logs
manifests
export_metadata
```

As fontes grandes, como cache do Hugging Face e cache de adaptadores baixados, ficam fora por padrão. Elas não são necessárias para entender a reprodução, porque os repositórios do Hugging Face usados podem ser baixados novamente.

In [ ]:
source_specs = []

if EXPORT_REQUIREMENTS:
    source_specs.append({
        "group": "requirements",
        "source": RPDCT_ROOT / "requirements-rpdct.txt",
        "archive_root": "pi-defense-rpdct/requirements-rpdct.txt",
        "required": False,
    })

if EXPORT_NOTEBOOKS:
    source_specs.append({
        "group": "notebooks",
        "source": NOTEBOOK_DIR,
        "archive_root": "pi-defense-rpdct/notebook",
        "required": True,
    })

if EXPORT_SAMPLE_DATA:
    source_specs.append({
        "group": "sample_data",
        "source": RPDCT_ROOT / "data" / "samples",
        "archive_root": "pi-defense-rpdct/data/samples",
        "required": False,
    })

if EXPORT_RESULTS:
    source_specs.append({
        "group": "results",
        "source": RPDCT_ROOT / "results" / RUN_MODE,
        "archive_root": f"pi-defense-rpdct/results/{RUN_MODE}",
        "required": False,
    })

if EXPORT_LOGS:
    source_specs.append({
        "group": "logs",
        "source": RPDCT_ROOT / "logs" / RUN_MODE,
        "archive_root": f"pi-defense-rpdct/logs/{RUN_MODE}",
        "required": False,
    })

if EXPORT_MANIFESTS:
    source_specs.append({
        "group": "manifests",
        "source": RPDCT_ROOT / "manifests" / RUN_MODE,
        "archive_root": f"pi-defense-rpdct/manifests/{RUN_MODE}",
        "required": False,
    })

if not EXPORT_HF_CACHE:
    source_specs.append({
        "group": "hf_cache",
        "source": RPDCT_ROOT / "hf_cache",
        "archive_root": "pi-defense-rpdct/hf_cache",
        "required": False,
        "disabled_by_policy": True,
        "policy_reason": "HF cache is intentionally excluded to avoid exporting large downloaded snapshots.",
    })

if not EXPORT_ADAPTER_CACHE:
    source_specs.append({
        "group": "adapter_cache",
        "source": RPDCT_ROOT / "adapter_cache",
        "archive_root": "pi-defense-rpdct/adapter_cache",
        "required": False,
        "disabled_by_policy": True,
        "policy_reason": "Adapter cache is intentionally excluded because adapters can be re-downloaded from Hugging Face.",
    })

if not EXPORT_VENV:
    source_specs.append({
        "group": "venv",
        "source": RPDCT_ROOT / ".venv",
        "archive_root": "pi-defense-rpdct/.venv",
        "required": False,
        "disabled_by_policy": True,
        "policy_reason": "Virtual environments are not portable and should not be exported.",
    })

pd.DataFrame(source_specs)

## 5. Construir índice de arquivos

Esta etapa percorre as fontes configuradas e cria três listas:

```text
file_records
missing_optional_sources
skipped_files
```

`file_records` contém os arquivos que entrarão no `.zip`.

`missing_optional_sources` registra fontes opcionais que não existem. Isso pode acontecer se a reprodução ainda não gerou resultados, logs ou manifestos.

`skipped_files` registra arquivos ou fontes ignoradas por política, como caches e ambiente virtual.

In [ ]:
file_records = []
missing_optional_sources = []
skipped_files = []

for spec in source_specs:
    group = spec["group"]
    source = Path(spec["source"])
    archive_root = spec["archive_root"]
    required = bool(spec.get("required", False))

    if spec.get("disabled_by_policy", False):
        skipped_files.append({
            "group": group,
            "path": str(source),
            "reason": spec.get("policy_reason", "disabled_by_policy"),
        })
        continue

    if not source.exists():
        record = {
            "group": group,
            "path": str(source),
            "reason": "source_missing",
            "required": required,
        }
        missing_optional_sources.append(record)
        if required:
            print("Aviso: fonte obrigatória ausente:", source)
        continue

    for path in iter_files(source):
        skip, reason = should_skip_file(path)
        if skip:
            skipped_files.append({
                "group": group,
                "path": str(path),
                "reason": reason,
            })
            continue

        if source.is_file():
            archive_path = archive_root
        else:
            relative = path.relative_to(source)
            archive_path = str(Path(archive_root) / relative)

        file_records.append({
            "group": group,
            "source_path": str(path),
            "archive_path": archive_path,
            "size_bytes": path.stat().st_size,
            "size_mb": path.stat().st_size / (1024 * 1024),
        })

file_index_df = pd.DataFrame(file_records)
missing_df = pd.DataFrame(missing_optional_sources)
skipped_df = pd.DataFrame(skipped_files)

print("Arquivos selecionados:", len(file_index_df))
print("Fontes opcionais ausentes:", len(missing_df))
print("Arquivos/fontes ignorados:", len(skipped_df))

display(file_index_df.head())

## 6. Gerar metadados leves de exportação

Os arquivos abaixo são gerados dentro de `exports/` e também entram no `.zip`:

```text
README.md
export_summary.json
export_manifest.json
export_manifest.md
index/file_index.csv
index/file_index.json
index/missing_optional_sources.json
index/skipped_files.json
```

Esses arquivos são pequenos e servem para documentar o pacote exportado sem duplicar os resultados grandes em uma pasta intermediária.

In [ ]:
created_at = utc_now()

total_size_bytes = int(file_index_df["size_bytes"].sum()) if len(file_index_df) else 0


def dataframe_records(df: pd.DataFrame) -> list[dict]:
    if df.empty:
        return []
    return df.to_dict(orient="records")

export_summary = {
    "export_name": "pi-defense-rpdct reproduction artifacts",
    "created_at_utc": created_at,
    "rpdct_root": str(RPDCT_ROOT),
    "run_mode": RUN_MODE,
    "export_root": str(EXPORT_ROOT),
    "zip_files_dir": str(ZIP_FILES_DIR),
    "archive_path": str(ARCHIVE_PATH),
    "export_mode": "zip_from_sources_no_intermediate_copy",
    "selected_files": len(file_index_df),
    "selected_size_bytes": total_size_bytes,
    "selected_size_mb": total_size_bytes / (1024 * 1024),
    "missing_optional_sources": len(missing_optional_sources),
    "skipped_files_or_sources": len(skipped_files),
    "policy": {
        "export_hf_cache": EXPORT_HF_CACHE,
        "export_adapter_cache": EXPORT_ADAPTER_CACHE,
        "export_venv": EXPORT_VENV,
        "max_file_size_mb": MAX_FILE_SIZE_MB,
    },
}

readme_text = f"""# pi-defense-rpdct reproduction artifact export

This folder contains lightweight metadata for the quick reproduction artifact export.

The export was generated at `{created_at}`.

## Export mode

This export uses a no-intermediate-copy strategy.

Large artifacts are not copied into `exports/`. Instead, this notebook indexes the original files and creates a zip archive directly from those source paths.

## Source root

```text
{RPDCT_ROOT}
```

## Archive

```text
{ARCHIVE_PATH}
```

## Included groups

The archive may include notebooks, requirements, sample data, results, logs and manifests when those files exist.

## Excluded by default

The export excludes `.venv/`, Hugging Face cache directories and adapter cache directories by default to avoid creating very large archives.

## Index files

- `index/file_index.csv`
- `index/file_index.json`
- `index/missing_optional_sources.json`
- `index/skipped_files.json`
"""

manifest = {
    "notebook": "02_export_reproduction_artifacts",
    "created_at_utc": created_at,
    "summary": export_summary,
    "source_specs": source_specs,
    "files": dataframe_records(file_index_df),
    "missing_optional_sources": missing_optional_sources,
    "skipped_files": skipped_files,
}

manifest_md = f"""# Manifesto da exportação da reprodução rápida

## Identificação

- Notebook: `02_export_reproduction_artifacts`
- Gerado em UTC: `{created_at}`
- Raiz da reprodução: `{RPDCT_ROOT}`
- Modo de execução: `{RUN_MODE}`

## Política

A exportação usa uma estratégia sem cópia intermediária pesada. O diretório `exports/` contém apenas metadados leves, enquanto o arquivo `.zip` é criado lendo diretamente os arquivos originais.

## Resumo

| Campo | Valor |
|---|---:|
| Arquivos selecionados | {len(file_index_df)} |
| Tamanho selecionado MB | {total_size_bytes / (1024 * 1024):.2f} |
| Fontes opcionais ausentes | {len(missing_optional_sources)} |
| Itens ignorados por política | {len(skipped_files)} |

## Arquivo compactado esperado

```text
{ARCHIVE_PATH}
```

## Observações

- `.venv/` não é exportado.
- `hf_cache/` não é exportado por padrão.
- `adapter_cache/` não é exportado por padrão.
- Arquivos ausentes em `missing_optional_sources.json` não indicam necessariamente erro; eles podem representar artefatos ainda não gerados.
"""

(EXPORT_ROOT / "README.md").write_text(readme_text, encoding="utf-8")
write_json(EXPORT_ROOT / "export_summary.json", export_summary)
write_json(EXPORT_ROOT / "export_manifest.json", manifest)
(EXPORT_ROOT / "export_manifest.md").write_text(manifest_md, encoding="utf-8")

file_index_df.to_csv(EXPORT_INDEX_DIR / "file_index.csv", index=False)
write_json(EXPORT_INDEX_DIR / "file_index.json", dataframe_records(file_index_df))
write_json(EXPORT_INDEX_DIR / "missing_optional_sources.json", missing_optional_sources)
write_json(EXPORT_INDEX_DIR / "skipped_files.json", skipped_files)

print("Metadados leves gerados em:", EXPORT_ROOT)

## 7. Adicionar os metadados gerados ao índice

Como os metadados foram criados depois do primeiro índice, esta etapa adiciona os próprios arquivos de `exports/` à lista de arquivos que entrarão no `.zip`.

Isso garante que o pacote final contenha tanto os artefatos originais quanto a documentação da exportação.

In [ ]:
metadata_records = []

for path in sorted(EXPORT_ROOT.rglob("*")):
    if not path.is_file():
        continue

    skip, reason = should_skip_file(path)
    if skip:
        skipped_files.append({
            "group": "export_metadata",
            "path": str(path),
            "reason": reason,
        })
        continue

    archive_path = str(Path("pi-defense-rpdct") / path.relative_to(RPDCT_ROOT))

    metadata_records.append({
        "group": "export_metadata",
        "source_path": str(path),
        "archive_path": archive_path,
        "size_bytes": path.stat().st_size,
        "size_mb": path.stat().st_size / (1024 * 1024),
    })

all_records = file_records + metadata_records
all_index_df = pd.DataFrame(all_records)

all_index_df.to_csv(EXPORT_INDEX_DIR / "file_index.csv", index=False)
write_json(EXPORT_INDEX_DIR / "file_index.json", all_index_df.to_dict(orient="records"))
write_json(EXPORT_INDEX_DIR / "skipped_files.json", skipped_files)

print("Total final de arquivos no índice:", len(all_index_df))
display(all_index_df.tail())

## 8. Criar arquivo compactado

Esta etapa cria o `.zip` final em `/workspace/zip_files/`.

O arquivo compactado é construído a partir dos caminhos originais listados no índice. Isso evita a duplicação de arquivos grandes em `exports/`.

Se já existir um `.zip` com o mesmo nome, ele será substituído.

In [ ]:
if len(all_index_df) == 0:
    raise RuntimeError("Nenhum arquivo selecionado para compactação.")

if ARCHIVE_PATH.exists():
    ARCHIVE_PATH.unlink()

with zipfile.ZipFile(
    ARCHIVE_PATH,
    mode="w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=6,
) as zip_file:
    for row in all_index_df.to_dict(orient="records"):
        source_path = Path(row["source_path"])
        archive_path = row["archive_path"]

        if not source_path.exists():
            skipped_files.append({
                "group": row.get("group"),
                "path": str(source_path),
                "reason": "source_missing_at_zip_time",
            })
            continue

        zip_file.write(source_path, archive_path)

archive_sha256 = sha256_file(ARCHIVE_PATH)
archive_size_bytes = ARCHIVE_PATH.stat().st_size

archive_info = {
    "archive_path": str(ARCHIVE_PATH),
    "archive_size_bytes": archive_size_bytes,
    "archive_size_mb": archive_size_bytes / (1024 * 1024),
    "archive_sha256": archive_sha256,
    "files_indexed": len(all_index_df),
    "created_at_utc": utc_now(),
}

write_json(EXPORT_ROOT / "archive_summary.json", archive_info)
write_json(ARCHIVE_INFO_PATH, archive_info)
write_json(EXPORT_INDEX_DIR / "skipped_files.json", skipped_files)

print("Arquivo compactado criado:", ARCHIVE_PATH)
print(f"Tamanho: {archive_info['archive_size_mb']:.2f} MB")
print("SHA256:", archive_sha256)

## 9. Conferência final

A conferência final valida se os principais arquivos de exportação foram criados.

O mais importante é que o `.zip` exista em `/workspace/zip_files/` e que os índices estejam disponíveis em `exports/reproduction_artifacts/<run_mode>/index/`.

In [ ]:
expected_outputs = {
    "readme": EXPORT_ROOT / "README.md",
    "export_summary": EXPORT_ROOT / "export_summary.json",
    "export_manifest_json": EXPORT_ROOT / "export_manifest.json",
    "export_manifest_md": EXPORT_ROOT / "export_manifest.md",
    "file_index_csv": EXPORT_INDEX_DIR / "file_index.csv",
    "file_index_json": EXPORT_INDEX_DIR / "file_index.json",
    "missing_optional_sources": EXPORT_INDEX_DIR / "missing_optional_sources.json",
    "skipped_files": EXPORT_INDEX_DIR / "skipped_files.json",
    "archive_summary": EXPORT_ROOT / "archive_summary.json",
    "archive_zip": ARCHIVE_PATH,
    "archive_info_external": ARCHIVE_INFO_PATH,
}

check_rows = []
for name, path in expected_outputs.items():
    check_rows.append({
        "name": name,
        "path": str(path),
        "exists": path.exists(),
        "size_bytes": path.stat().st_size if path.exists() else None,
    })

check_df = pd.DataFrame(check_rows)
display(check_df)

missing_required = check_df.loc[~check_df["exists"], "name"].tolist()
if missing_required:
    raise RuntimeError("Arquivos esperados ausentes: " + ", ".join(missing_required))

print("Exportação da reprodução rápida concluída com sucesso.")
print("Zip:", ARCHIVE_PATH)

## 10. Próximos passos

Depois de executar este notebook, o arquivo principal para compartilhar ou arquivar é:

```text
/workspace/zip_files/pi_defense_rpdct_artifacts_quick.zip
```

Os metadados leves da exportação ficam em:

```text
/workspace/pi-defense-rpdct/exports/reproduction_artifacts/quick/
```

Antes de subir o `.zip` para algum lugar, vale conferir:

```text
index/file_index.csv
index/missing_optional_sources.json
index/skipped_files.json
archive_summary.json
```

Esses arquivos mostram exatamente o que entrou na exportação, o que não existia e o que foi ignorado por política de espaço.